# CyberRange OS - QLoRA Fine-Tune (Colab / Kaggle T4)

Runtime -> Change runtime type -> **T4 GPU**. Then upload your `dataset.jsonl`
(produced by `build_dataset.py`) when prompted, and Run All.

Output: `cyberrange-sec.gguf` (q4_k_m) ready for `ollama create`.

In [ ]:
# 1. Install Unsloth (fast QLoRA + GGUF export)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Load a small base model in 4-bit. Swap MODEL for Llama-3.2-3B if you prefer.
from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  # or unsloth/Llama-3.2-3B-Instruct-bnb-4bit
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)

# 3. Attach LoRA adapters (only these train)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

In [ ]:
# 4. Upload dataset.jsonl (chat format) and apply the model's chat template
from google.colab import files  # on Kaggle, place dataset.jsonl in /kaggle/working instead
up = files.upload()

from datasets import load_dataset
ds = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_chat(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(format_chat)
print(ds[0]["text"][:500])
print("examples:", len(ds))

In [ ]:
# 5. Train (QLoRA SFT). ~1-3 GPU-hours depending on dataset size.
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=42, output_dir="outputs",
    ),
)
trainer.train()

In [ ]:
# 6. Quick sanity check
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"Alert: repeated SQL injection signatures from one IP to the web server. Triage as JSON."}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tokenizer.decode(model.generate(input_ids=inputs, max_new_tokens=200)[0], skip_special_tokens=True))

In [ ]:
# 7. Merge + export to GGUF (q4_k_m) and download
model.save_pretrained_gguf("cyberrange-sec", tokenizer, quantization_method="q4_k_m")
import glob, shutil
gguf = glob.glob("cyberrange-sec/*.gguf")[0]
shutil.copy(gguf, "cyberrange-sec.gguf")
print("GGUF:", gguf)
from google.colab import files
files.download("cyberrange-sec.gguf")